<a href="https://colab.research.google.com/github/lxmfran/bugs_in_the_fly/blob/main/shiu_simulator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ATTENTION: only run this cell when on google colab
!git clone https://github.com/philshiu/Drosophila_brain_model.git
!pip install brian2
%cd Drosophila_brain_model

Cloning into 'Drosophila_brain_model'...
remote: Enumerating objects: 183, done.
remote: Counting objects: 100% (88/88), done.
remote: Compressing objects: 100% (21/21), done.
remote: Total 183 (delta 75), reused 67 (delta 67), pack-reused 95 (from 1)
Receiving objects: 100% (183/183), 185.05 MiB | 28.08 MiB/s, done.
Resolving deltas: 100% (83/83), done.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 40.0 MB/s  0:00:00
/content/Drosophila_brain_model


In [2]:
from model import run_exp
from model import default_params as params
import utils as utl
from brian2 import Hz

config = {
    'path_res'  : './results/example',                              # directory to store results
    'path_comp' : './2023_03_23_completeness_630_final.csv',        # csv of the complete list of Flywire neurons
    'path_con'  : './2023_03_23_connectivity_630_final.parquet',    # connectivity data
    'n_proc'    : -1,                                               # number of CPU cores (-1: use all)
}

# Introduction
## Underlying connectivity data
The connectivity of the fly brain is stored in the folowing files:
- neurons present: `config['path_comp']`
- connectivity between neurons: `config['path_con]`

## Model parameters
The equation and constants for the leaky integrate and fire model are defined
in the dictionary `default_params` in the beginning of the file `model.py`:

```
default_params = {
    # trials
    't_run'     : 1000 * ms,              # duration of trial
    'n_run'     : 30,                     # number of runs

    'v_0'       : -52 * mV,               # resting potential
    'v_rst'     : -52 * mV,               # reset potential after spike
    [...]
```
We can also change values
and pass the modified dictionary to the model (see Experiment 1).

## Addressing neurons
Here, we want to stimulate some sugar-sensing neurons in the right hemisphere.
The neurons of interest are defined via their flywire IDs:

In [3]:
neu_sugar = [
    720575940624963786,
    720575940630233916,
    720575940637568838,
    720575940638202345,
    720575940617000768,
    720575940630797113,
    720575940632889389,
    720575940621754367,
    720575940621502051,
    720575940640649691,
    720575940639332736,
    720575940616885538,
    720575940639198653,
    720575940620900446,
    720575940617937543,
    720575940632425919,
    720575940633143833,
    720575940612670570,
    720575940628853239,
    720575940629176663,
    720575940611875570,
]

For an easier identification, we define also a mapping from the flywire IDs to custom
names. The above neurons are calles `sugar_1`, `sugar_2` etc:

In [4]:
flyid2name = { f: f'sugar_{i+1}' for i, f in enumerate(neu_sugar) }
flyid2name

{720575940624963786: 'sugar_1',
 720575940630233916: 'sugar_2',
 720575940637568838: 'sugar_3',
 720575940638202345: 'sugar_4',
 720575940617000768: 'sugar_5',
 720575940630797113: 'sugar_6',
 720575940632889389: 'sugar_7',
 720575940621754367: 'sugar_8',
 720575940621502051: 'sugar_9',
 720575940640649691: 'sugar_10',
 720575940639332736: 'sugar_11',
 720575940616885538: 'sugar_12',
 720575940639198653: 'sugar_13',
 720575940620900446: 'sugar_14',
 720575940617937543: 'sugar_15',
 720575940632425919: 'sugar_16',
 720575940633143833: 'sugar_17',
 720575940612670570: 'sugar_18',
 720575940628853239: 'sugar_19',
 720575940629176663: 'sugar_20',
 720575940611875570: 'sugar_21'}

# Running simulations
## Activating a set of neurons
To run a simulation exciting these nerons we have to call `run_exp` supplying the following:
- unique name for the simulation: `exp_name`
- a list of neurons we want to stimulate: `neu_sugar`
- the connectivity data: `config['path_comp']` and `config['path_con]`
- path to store the output: `config['path_res']`
- number of CPU cores use: `config['n_procs]`

Note that running this on Google Colab can take roughly 20 minutes; it is substantially faster on a local install, depending on the number of CPU cores. By default, the neurons are excited at 200 Hz.

In [ ]:
# activate sugar sensing neurons
run_exp(exp_name='sugarR', neu_exc=neu_sugar, **config)

The `.parquet` file created during a simulation contains all spikes events of all neurons in the model.
We load the data again from disk by passing a list of result files to the `utl.load_exps` function.

We can see from the size of the dataframe
that more than 400 000 spikes were generated by activating the sugar neurons (30 trials, 1 s each).

In [ ]:
# load data from disk
df_spike = utl.load_exps([ './results/example/sugarR.parquet' ])
df_spike

The spike times can be converted to spike rates [Hz] via `utl.get_rate`, which requires the duration of each trial.
`utl.get_rate` returns `pandas.DataFrame` objects:
1. spike rate for each neuron (rows) in each experiment (column): `df_rate`
2. standard deviation of rate across trials: `df_rate_std`

For convenience, we can optionally pass the `flyid2name` dictionary to `utl.get_rate` in order to convert flywire IDs into
meaningful names.

We can see that only about 400 neurons show activity during the simulations.

In [ ]:
# calculate spike rate and standard deviation
df_rate, df_rate_std = utl.get_rate(df_spike, t_run=params['t_run'], n_run=params['n_run'], flyid2name=flyid2name)
# sort by spike rate
df_rate.sort_values('sugarR', ascending=False)

## Change stimulation frequency

We want to change the frequency of the stimulation of the sugar neurons.
To do so we modify the value for `r_poi` in the `default_params` dictionary and pass the altered dictionary to the `run_exp` function.

Note: Since physical quantities in `brian2` have to have the correct unit, we also need the `brian2.Hz` object
to define a frequency.

In [5]:
# run with different frequency
params['r_poi'] = 100 * Hz

run_exp(exp_name='sugarR_100Hz', neu_exc=neu_sugar, params=params, **config)

>>> Skipping experiment sugarR_100Hz because results/example/sugarR_100Hz.parquet exists and force_overwrite = False


In [6]:
df_spike = utl.load_exps(['./results/example/sugarR_100Hz.parquet'])
df_rate, df_rate_std = utl.get_rate(df_spike, t_run=params['t_run'], n_run=params['n_run'], flyid2name=flyid2name)

id_mn9 = 720575940660219265
print("MN9 media (Hz):", df_rate.loc[id_mn9, "sugarR_100Hz"])
print("MN9 desviación entre ensayos (Hz):", df_rate_std.loc[id_mn9, "sugarR_100Hz"])

MN9 media (Hz): 67.03333333333333
MN9 desviación entre ensayos (Hz): 6.595368745482612


In [7]:
import pandas as pd

df_con = pd.read_parquet(config['path_con'])
print(df_con.shape)
print(df_con.columns.tolist())
df_con.head()

(14687178, 7)
['Presynaptic_ID', 'Postsynaptic_ID', 'Presynaptic_Index', 'Postsynaptic_Index', 'Connectivity', 'Excitatory', 'Excitatory x Connectivity']


,Presynaptic_ID,Postsynaptic_ID,Presynaptic_Index,Postsynaptic_Index,Connectivity,Excitatory,Excitatory x Connectivity
0,720575940596125868,720575940640198784,0,119081,1,1,1
1,720575940596125868,720575940605825666,0,2507,2,1,2
2,720575940596125868,720575940620159366,0,40129,10,1,10
3,720575940596125868,720575940622736417,0,54129,2,1,2
4,720575940596125868,720575940630993423,0,94914,1,1,1


In [8]:
id_mn9 = 720575940660219265

entradas = df_con[df_con['Postsynaptic_ID'] == id_mn9]
entradas = entradas[['Presynaptic_ID', 'Connectivity', 'Excitatory']]
entradas = entradas.sort_values('Connectivity', ascending=False)

print("Neuronas que envían sinapsis a MN9:", len(entradas))
print("Sinapsis totales que recibe MN9:", entradas['Connectivity'].sum())
print(entradas['Excitatory'].value_counts())
entradas.head(15)

Neuronas que envían sinapsis a MN9: 229
Sinapsis totales que recibe MN9: 5591
Excitatory
-1    122
 1    107
Name: count, dtype: int64


,Presynaptic_ID,Connectivity,Excitatory
4135,720575940619853515,561,1
9411,720575940636809646,448,-1
10756,720575940623211725,424,1
1686,720575940632252743,281,1
3405,720575940620156209,267,-1
14853,720575940641281013,257,-1
4223,720575940626961241,185,-1
7259,720575940618028125,183,-1
1744,720575940649829241,173,-1
10572,720575940611167842,155,-1


In [9]:
cand = entradas.set_index('Presynaptic_ID')
cand['Hz_azucar'] = df_rate['sugarR_100Hz'].reindex(cand.index).fillna(0)

activas = cand[cand['Hz_azucar'] > 0].sort_values('Hz_azucar', ascending=False)
print("Entradas de MN9 activas con azúcar:", len(activas), "de", len(cand))
activas

Entradas de MN9 activas con azúcar: 62 de 229


,Connectivity,Excitatory,Hz_azucar
Presynaptic_ID,,,
720575940615041430,1,-1,95.733333
720575940615671106,2,1,61.700000
720575940619877396,32,1,58.633333
720575940631918649,4,-1,58.533333
720575940636053678,4,-1,50.466667
...,...,...,...
720575940626860421,10,-1,0.366667
720575940620297819,1,-1,0.233333
720575940617497437,1,-1,0.166667


In [10]:
print("¿Alguna neurona del azúcar conecta con MN9?", set(neu_sugar) & set(entradas['Presynaptic_ID']))

activas['empuje'] = activas['Hz_azucar'] * activas['Connectivity'] * activas['Excitatory']
activas.sort_values('empuje', ascending=False).head(10)

¿Alguna neurona del azúcar conecta con MN9? set()


,Connectivity,Excitatory,Hz_azucar,empuje
Presynaptic_ID,,,,
720575940623211725,424,1,45.666667,19362.666667
720575940619853515,561,1,28.466667,15969.800000
720575940632252743,281,1,44.166667,12410.833333
720575940609407939,88,1,32.266667,2839.466667
720575940627847752,74,1,31.033333,2296.466667
720575940619877396,32,1,58.633333,1876.266667
720575940616857174,25,1,32.433333,810.833333
720575940619114048,20,1,17.700000,354.000000
720575940621261296,11,1,25.133333,276.466667


In [12]:
id_mn9 = 720575940660219265
id_silenciar = 720575940623211725

run_exp(exp_name='sugarR_100Hz_silenciada', neu_exc=neu_sugar, neu_slnc=[id_silenciar], params=params, **config)

>>> Experiment:     sugarR_100Hz_silenciada
    Output file:    results/example/sugarR_100Hz_silenciada.parquet
    Excited neurons: 21
    Silenced neurons: 1


WARNING    /usr/local/lib/python3.13/dist-packages/joblib/externals/loky/process_executor.py:787: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
 [py.warnings]
  warnings.warn(



    Elapsed time:   1056 s


In [13]:
df_spike = utl.load_exps(['./results/example/sugarR_100Hz.parquet',
                          './results/example/sugarR_100Hz_silenciada.parquet'])
df_rate2, df_std2 = utl.get_rate(df_spike, t_run=params['t_run'], n_run=params['n_run'])

base = df_rate2.loc[id_mn9, 'sugarR_100Hz']
sil  = df_rate2.loc[id_mn9, 'sugarR_100Hz_silenciada']
print(f"MN9 normal:     {base:.1f} Hz  (± {df_std2.loc[id_mn9, 'sugarR_100Hz']:.1f})")
print(f"MN9 silenciada: {sil:.1f} Hz  (± {df_std2.loc[id_mn9, 'sugarR_100Hz_silenciada']:.1f})")
print(f"Cambio: {100*(sil-base)/base:+.1f} %   (umbral de Shiu: -20 %)")

MN9 normal:     67.0 Hz  (± 6.6)
MN9 silenciada: 50.7 Hz  (± 5.0)
Cambio: -24.3 %   (umbral de Shiu: -20 %)


We load the results via the `utl.load_exps` function and convert the spike events to rates with `utl.get_rate`

In [ ]:
ps = [
    './results/example/sugarR.parquet',
    './results/example/sugarR_100Hz.parquet',
]

df_spike = utl.load_exps(ps)
df_rate, df_rate_std = utl.get_rate(df_spike, t_run=params['t_run'], n_run=params['n_run'], flyid2name=flyid2name)
df_rate.sort_values('sugarR_100Hz', ascending=False, inplace=True)
df_rate

## Silencing neurons
We want to silence the most active neurons individually to see how that changes the activity patterns.
We do so by passing the neuron IDs we want to silence as a list `run_exp` via the `neu_slnc` argument.
In the following example, we are silencing a single neuron `[ i ]` while exciting the sugar neurons `neu_sugar`.
We can then investigate how silencing of each individual neuron affects the firing rate of a given neuron, say, MN9.

In [ ]:
#First, let's check on the MN9 firing rate when no neurons are silenced.
id_mn9 = 720575940660219265 #id for MN9
x = df_rate.loc[id_mn9, "sugarR_100Hz"]
print(f'Rate for neuron {id_mn9} is {x}')

In [ ]:
# IDs of 3 most active neurons. These neurons are all sugar-sensing neurons.
ids = df_rate.sort_values('sugarR_100Hz', ascending=False).index[:3]

for i in ids:
    run_exp(exp_name=f'sugarR-{i}', neu_exc=neu_sugar, neu_slnc=[ i ], params=params, **config)

In [ ]:
# output files
ps = [ f'./results/example/sugarR-{i}.parquet' for i in ids ]

# calculate spike rate and sort
df_spike = utl.load_exps(ps)
df_rate, df_rate_std = utl.get_rate(df_spike, t_run=params['t_run'], n_run=params['n_run'])
df_rate.loc[id_mn9, :].sort_values(ascending=True)